# 2. Fine-tune VideoMAE to confirm accidents (stage 2 of the incident pipeline)

Stage 1 (trajectory rules) flags candidate events; this classifier looks at a 16-frame clip around the
event and outputs P(accident), cutting false alarms.

**Data:** CADP, DoTA or the AI City Challenge accident track — check each licence first. Cut clips of 2–4 s
and arrange them on Drive as:

```
MyDrive/accident_clips/{train,val,test}/{accident,normal}/*.mp4
```
"Normal" clips should include hard negatives: heavy braking at signals, dense two-wheeler filtering, occlusions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install transformers==4.46.3 accelerate decord evaluate

In [ ]:
import glob, os, numpy as np, torch, decord
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification, TrainingArguments, Trainer
ROOT = '/content/drive/MyDrive/accident_clips'
OUT = '/content/drive/MyDrive/traffic_ai_runs/videomae_accident'
CKPT = 'MCG-NJU/videomae-base-finetuned-kinetics'   # VideoMAE V2 (OpenGVLab) works too with trust_remote_code
labels = ['normal', 'accident']
proc = VideoMAEImageProcessor.from_pretrained(CKPT)
model = VideoMAEForVideoClassification.from_pretrained(
    CKPT, num_labels=2, id2label=dict(enumerate(labels)), label2id={l: i for i, l in enumerate(labels)},
    ignore_mismatched_sizes=True)
N = model.config.num_frames

class Clips(torch.utils.data.Dataset):
    def __init__(self, split, train=False):
        self.items = [(p, i) for i, l in enumerate(labels) for p in glob.glob(f'{ROOT}/{split}/{l}/*.mp4')]
        self.train = train
    def __len__(self): return len(self.items)
    def __getitem__(self, k):
        p, y = self.items[k]
        vr = decord.VideoReader(p)
        idx = np.linspace(0, len(vr) - 1, N).astype(int)
        if self.train:  # temporal jitter
            idx = np.clip(idx + np.random.randint(-2, 3, N), 0, len(vr) - 1)
        frames = list(vr.get_batch(idx).asnumpy())
        if self.train and np.random.rand() < 0.5:
            frames = [f[:, ::-1].copy() for f in frames]
        x = proc(frames, return_tensors='pt')['pixel_values'][0]
        return {'pixel_values': x, 'labels': y}

train_ds, val_ds, test_ds = Clips('train', True), Clips('val'), Clips('test')
print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score
def metrics(p):
    y = p.label_ids; prob = torch.tensor(p.predictions).softmax(-1)[:, 1].numpy(); yhat = prob >= 0.5
    return {'recall': recall_score(y, yhat), 'precision': precision_score(y, yhat, zero_division=0),
            'f1': f1_score(y, yhat), 'false_alarm_rate': float(((yhat == 1) & (y == 0)).sum() / max((y == 0).sum(), 1))}
args = TrainingArguments(OUT, per_device_train_batch_size=4, per_device_eval_batch_size=4, gradient_accumulation_steps=4,
                         learning_rate=5e-5, num_train_epochs=8, warmup_ratio=0.1, eval_strategy='epoch',
                         save_strategy='epoch', load_best_model_at_end=True, metric_for_best_model='f1',
                         fp16=True, logging_steps=20, remove_unused_columns=False, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=metrics)
trainer.train()
print(trainer.evaluate(test_ds))

In [ ]:
# Save for the pipeline:  python -m traffic_ai.cli video --source clip.mp4 --videomae <this folder>
trainer.save_model(f'{OUT}/final'); proc.save_pretrained(f'{OUT}/final')